# 01 · Data checks

This notebook loads the six raw ADEM CSVs in full and checks how the data is structured before any cleaning. It does not change the raw files.

One row is one vacancy–skill pair: a vacancy with five skills appears on five rows.

Checks:

1. First look: columns, types and a few rows of each file
2. Total rows and unique vacancies per file and per year
3. Which months are covered, especially for 2026
4. Whether each vacancy has the same `positions`, canton, occupation and sector on all its rows
5. Whether any `vacancy_id` appears in more than one file
6. Exact duplicate rows and real missing-value rates

The results decide the cleaning rules used in `02_clean_data.ipynb`. Loading all files can take a few minutes.


## 0. Load all files

In [2]:
from pathlib import Path
import pandas as pd
from pandas.api.types import union_categoricals
from IPython.display import display

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)

# Support running from either the project root or notebooks/.
project_dir = Path.cwd()
if not (project_dir / 'data' / 'raw').is_dir():
    project_dir = project_dir.parent

raw_dir = project_dir / 'data' / 'raw'
files = sorted(raw_dir.glob('*.csv'))
if not files:
    raise FileNotFoundError(f'No CSV files found in {raw_dir.resolve()}')

TEXT_COLS = ['skill', 'skill_uri', 'canton', 'occupation_code',
             'occupation_label', 'nace_code', 'nace_label']
DTYPES = {'vacancy_id': 'int64', 'positions': 'int32',
          'month': 'int8', 'year': 'int16'}
DTYPES.update({col: 'category' for col in TEXT_COLS})
ORIGINAL_COLS = ['vacancy_id', 'skill', 'skill_uri', 'positions', 'month', 'year',
                 'canton', 'occupation_code', 'occupation_label', 'nace_code', 'nace_label']

frames = []
for file in files:
    part = pd.read_csv(file, dtype=DTYPES)
    missing_cols = set(ORIGINAL_COLS) - set(part.columns)
    extra_cols = set(part.columns) - set(ORIGINAL_COLS)
    if missing_cols or extra_cols:
        print(f'WARNING {file.name}: missing {missing_cols}, extra {extra_cols}')
    part['source_file'] = file.stem.replace('datasc-skills-vacancies-', '')
    frames.append(part)
    print(f'{file.name}: {len(part):,} rows')

# Give every text column the same categories in all files so they combine cleanly.
for col in TEXT_COLS:
    cats = union_categoricals([f[col] for f in frames]).categories
    for f in frames:
        f[col] = f[col].cat.set_categories(cats)

df = pd.concat(frames, ignore_index=True)
df['source_file'] = df['source_file'].astype('category')
del frames

print(f'\nTotal: {len(df):,} rows, {df.shape[1]} columns')
print(f'Memory used: {df.memory_usage(deep=True).sum() / 1e9:.2f} GB')

datasc-skills-vacancies-2015-2016.csv: 271,257 rows
datasc-skills-vacancies-2017-2018.csv: 424,157 rows
datasc-skills-vacancies-2019-2020.csv: 431,048 rows
datasc-skills-vacancies-2021-2022.csv: 632,035 rows
datasc-skills-vacancies-2023-2024.csv: 581,239 rows
datasc-skills-vacancies-2025-2026.csv: 545,357 rows

Total: 2,885,093 rows, 12 columns
Memory used: 0.08 GB


## 1. First look

Columns, types and the first rows of each file, to see what one row contains.

In [3]:
print('Columns:', ORIGINAL_COLS)
display(df[ORIGINAL_COLS].dtypes.astype(str).rename('type').to_frame())

for name, part in df.groupby('source_file', observed=True):
    print(f'\n{name}: first 3 rows')
    display(part[ORIGINAL_COLS].head(3))

Columns: ['vacancy_id', 'skill', 'skill_uri', 'positions', 'month', 'year', 'canton', 'occupation_code', 'occupation_label', 'nace_code', 'nace_label']


,type
vacancy_id,int64
skill,category
skill_uri,category
positions,int32
month,int8
year,int16
canton,category
occupation_code,category
occupation_label,category
nace_code,category



2015-2016: first 3 rows


,vacancy_id,skill,skill_uri,positions,month,year,canton,occupation_code,occupation_label,nace_code,nace_label
0,-245188796,NaN,NaN,1,7,2015,Luxembourg,F1106,Ingénierie et études du BTP,O,ADMINISTRATION PUBLIQUE
1,-747184792,use gardening equipment,http://data.europa.eu/esco/skill/8a1942a8-bebd...,1,12,2016,Luxembourg,A1203,Entretien des espaces verts,O,ADMINISTRATION PUBLIQUE
2,-143211672,plan,http://data.europa.eu/esco/skill/e49f4158-9d4c...,1,12,2016,Diekirch,A1204,Protection du patrimoine naturel,O,ADMINISTRATION PUBLIQUE



2017-2018: first 3 rows


,vacancy_id,skill,skill_uri,positions,month,year,canton,occupation_code,occupation_label,nace_code,nace_label
271257,-1049353367,advise others,http://data.europa.eu/esco/skill/045f71e6-0699...,1,2,2017,Esch-sur-Alzette,A1203,Entretien des espaces verts,G,COMMERCE; RÉPARATION D'AUTOMOBILES ET DE MOTOC...
271258,-1049353367,sales activities,http://data.europa.eu/esco/skill/450de541-797a...,1,2,2017,Esch-sur-Alzette,A1203,Entretien des espaces verts,G,COMMERCE; RÉPARATION D'AUTOMOBILES ET DE MOTOC...
271259,-1049353367,French,http://data.europa.eu/esco/skill/e747e77e-0ea1...,1,2,2017,Esch-sur-Alzette,A1203,Entretien des espaces verts,G,COMMERCE; RÉPARATION D'AUTOMOBILES ET DE MOTOC...


... (4 outputs left out)


2023-2024: first 3 rows


,vacancy_id,skill,skill_uri,positions,month,year,canton,occupation_code,occupation_label,nace_code,nace_label
1758497,-831324346,mobility disability,http://data.europa.eu/esco/skill/842d7168-c18e...,1,1,2023,Esch-sur-Alzette,G1603,Personnel polyvalent en restauration,I,HÉBERGEMENT ET RESTAURATION
1758498,-831324346,sales activities,http://data.europa.eu/esco/skill/450de541-797a...,1,1,2023,Esch-sur-Alzette,G1603,Personnel polyvalent en restauration,I,HÉBERGEMENT ET RESTAURATION
1758499,-831324346,apply HACCP,http://data.europa.eu/esco/skill/5ba1d5b9-3f8d...,1,1,2023,Esch-sur-Alzette,G1603,Personnel polyvalent en restauration,I,HÉBERGEMENT ET RESTAURATION



2025-2026: first 3 rows


,vacancy_id,skill,skill_uri,positions,month,year,canton,occupation_code,occupation_label,nace_code,nace_label
2339736,-414383751,SAP Data Services,http://data.europa.eu/esco/skill/f8e3425c-fe44...,1,3,2025,Luxembourg,M1602,Opérations administratives,O,ADMINISTRATION PUBLIQUE
2339737,-414383751,computer technology,http://data.europa.eu/esco/skill/ddc3119d-1d6e...,1,3,2025,Luxembourg,M1602,Opérations administratives,O,ADMINISTRATION PUBLIQUE
2339738,-414383751,computer science,http://data.europa.eu/esco/skill/7b5cce4d-c7fe...,1,3,2025,Luxembourg,M1602,Opérations administratives,O,ADMINISTRATION PUBLIQUE


## 2. Rows, vacancies and posts

- **rows**: vacancy–skill pairs
- **vacancies**: unique `vacancy_id`
- **posts**: `positions` counted once per vacancy (first row), not summed over every row

In [4]:
# One row per vacancy within each file, used to count vacancies and posts.
vac = df.drop_duplicates(['source_file', 'vacancy_id'])

by_file = pd.DataFrame({
    'rows': df.groupby('source_file', observed=True).size(),
    'vacancies': vac.groupby('source_file', observed=True).size(),
    'posts': vac.groupby('source_file', observed=True)['positions'].sum(),
})
by_file['rows_per_vacancy'] = (by_file['rows'] / by_file['vacancies']).round(2)
print('By file:')
display(by_file)

# Per year. A vacancy is counted in each year it appears in.
vac_year = df.drop_duplicates(['source_file', 'vacancy_id', 'year'])
by_year = pd.DataFrame({
    'rows': df.groupby('year').size(),
    'vacancies': vac_year.groupby('year').size(),
    'posts': vac_year.groupby('year')['positions'].sum(),
})
by_year['rows_per_vacancy'] = (by_year['rows'] / by_year['vacancies']).round(2)
print('By year:')
display(by_year)

# Check that each file only contains the years in its name.
print('Years found in each file:')
display(df.groupby('source_file', observed=True)['year'].unique())

By file:


,rows,vacancies,posts,rows_per_vacancy
source_file,,,,
2015-2016,271257,46301,65416,5.86
2017-2018,424157,56405,76520,7.52
2019-2020,431048,51863,73776,8.31
2021-2022,632035,67012,92808,9.43
2023-2024,581239,60088,78161,9.67
2025-2026,545357,49352,62806,11.05


By year:


,rows,vacancies,posts,rows_per_vacancy
year,,,,
2015,118117,21672,31404,5.45
2016,153140,24629,34012,6.22
2017,196113,27104,35738,7.24
2018,228044,29301,40782,7.78
2019,227218,28175,39451,8.06
2020,203830,23688,34325,8.60
2021,302572,32629,44994,9.27
2022,329463,34383,47814,9.58
2023,280051,30003,40135,9.33


Years found in each file:


source_file
2015-2016    [2015, 2016]
2017-2018    [2017, 2018]
2019-2020    [2020, 2019]
2021-2022    [2021, 2022]
2023-2024    [2023, 2024]
2025-2026    [2025, 2026]
Name: year, dtype: object

## 3. Months covered

Unique vacancies per year and month. Empty cells or very low counts point to missing or incomplete months. Check 2026 closely before comparing it with full years.

In [5]:
vac_month = df.drop_duplicates(['source_file', 'vacancy_id', 'year', 'month'])
month_table = pd.crosstab(vac_month['year'], vac_month['month'])
display(month_table)

print('Months with no vacancies (within the years present):')
gaps = [(y, m) for y in month_table.index for m in range(1, 13)
        if m not in month_table.columns or month_table.loc[y, m] == 0]
print(gaps if gaps else 'None')

# Values outside 1-12 would indicate bad data.
bad_months = df.loc[~df['month'].between(1, 12), 'month'].unique()
print('Invalid month values:', bad_months if len(bad_months) else 'None')

month,1,2,3,4,5,6,7,8,9,10,11,12
year,,,,,,,,,,,,
2015,1827,1786,2040,1808,1752,1923,1894,1534,1927,1911,1730,1540
2016,2147,2353,2304,2020,1987,2082,2021,1757,2326,2063,1918,1651
2017,2287,2445,2496,2120,2699,2178,2202,2028,2473,2361,2129,1686
2018,2778,2320,2775,2385,2664,2690,2575,1978,2354,2732,2158,1892
2019,2524,2388,2721,2470,2570,2407,2624,1902,2356,2362,2196,1715
2020,2841,2596,1586,1086,1436,2031,2140,1639,2391,2273,1912,1757
2021,2192,2411,2912,2371,2873,3114,2871,2549,3070,3012,2704,2550
2022,3015,2820,3718,3186,3468,3187,2998,2717,2809,2648,2518,2062
2023,3142,2616,3008,2369,2687,2746,2526,2266,2415,2457,2066,1705


Months with no vacancies (within the years present):
[(2026, 11), (2026, 12)]
Invalid month values: None


## 4. Are vacancy details the same on all rows?

For each vacancy (within a file), count how many distinct values each column takes. Missing values count as a distinct value. If a vacancy is consistent, every column should have exactly 1.

In [6]:
CHECK_COLS = ['positions', 'month', 'year', 'canton', 'occupation_code', 'nace_code']

distinct = (df.groupby(['source_file', 'vacancy_id'], observed=True)[CHECK_COLS]
              .nunique(dropna=False))

inconsistent = (distinct > 1)
summary = pd.DataFrame({
    'vacancies_with_more_than_one_value': inconsistent.sum(),
    'pct_of_vacancies': (inconsistent.mean() * 100).round(3),
})
display(summary)

print('By file (number of inconsistent vacancies per column):')
display(inconsistent.groupby(level='source_file', observed=True).sum())

,vacancies_with_more_than_one_value,pct_of_vacancies
positions,308,0.093
month,1505,0.455
year,0,0.000
canton,917,0.277
occupation_code,1479,0.447
nace_code,1381,0.417


By file (number of inconsistent vacancies per column):


,positions,month,year,canton,occupation_code,nace_code
source_file,,,,,,
2015-2016,0,0,0,0,0,0
2017-2018,0,0,0,0,0,0
2019-2020,10,60,0,38,56,49
2021-2022,166,763,0,456,746,704
2023-2024,0,0,0,0,0,0
2025-2026,132,682,0,423,677,628


In [7]:
# Show a few examples for each column that has inconsistencies.
for col in CHECK_COLS:
    bad_ids = distinct.index[distinct[col] > 1]
    if len(bad_ids) == 0:
        continue
    print(f'\nExamples where {col} differs within a vacancy:')
    sample_keys = bad_ids[:3]
    example = df.set_index(['source_file', 'vacancy_id']).loc[sample_keys]
    display(example[['skill', 'positions', 'month', 'year', 'canton',
                     'occupation_code', 'nace_code']].head(20))


Examples where positions differs within a vacancy:


skill  positions  month  year            canton occupation_code nace_code
source_file vacancy_id                                                                                                   
2019-2020   -1015849354  cooperate with professionals          1      5  2019          Capellen           F1106         M
            -1015849354                   mathematics          1      5  2019          Capellen           F1106         M
            -1015849354                 work in teams          1      5  2019          Capellen           F1106         M
            -1015849354                         logic          1      5  2019          Capellen           F1106         M
            -1015849354                 communication          1      5  2019          Capellen           F1106         M
            -1015849354       ICT networking hardware          1      5  2019          Capellen           F1106         M
            -1015849354     monitor construction site          1      5  2019          Capellen           F1106         M
            -1015849354                 advise others          1      5  2019          Capellen           F1106         M
            -1015849354         construction industry          1      5  2019          Capellen           F1106         M
            -1015849354   work in an organised manner          1      5  2019          Capellen           F1106         M
            -1015849354                        French          7      1  2019  Esch-sur-Alzette           N4301         H
            -1015849354     use presentation software          7      1  2019  Esch-sur-Alzette           N4301         H
            -1015849354          security regulations          7      1  2019  Esch-sur-Alzette           N4301         H
            -1015849354            work independently          7      1  2019  Esch-sur-Alzette           N4301         H
            -1015849354                work in shifts          7      1  2019  Esch-sur-Alzette           N4301         H
            -244088458           use microsoft office          1      1  2019           Redange           K2104         P
            -244088458   use word processing software          1      1  2019           Redange           K2104         P
            -244088458      use spreadsheets software          1      1  2019           Redange           K2104         P
            -244088458   use word processing software          2      5  2019        Luxembourg           M1203         O
            -244088458              make reservations          2      5  2019        Luxembourg           M1203         O


Examples where month differs within a vacancy:


skill  positions  month  year      canton occupation_code nace_code
source_file vacancy_id                                                                                                            
2019-2020   -1015854730                         use microsoft office          1      1  2019  Luxembourg           M1204         K
            -1015854730                          computer technology          1      1  2019  Luxembourg           M1204         K
            -1015854730                    use presentation software          1      1  2019  Luxembourg           M1204         K
            -1015854730               geographic information systems          1      1  2019  Luxembourg           M1204         K
            -1015854730                        product comprehension          1      1  2019  Luxembourg           M1204         K
            -1015854730                                assertiveness          1      1  2019  Luxembourg           M1204         K
            -1015854730                                   accounting          1      1  2019  Luxembourg           M1204         K
            -1015854730                                    economics          1      1  2019  Luxembourg           M1204         K
            -1015854730                                  lead others          1      1  2019  Luxembourg           M1204         K
            -1015854730                                 report facts          1      1  2019  Luxembourg           M1204         K
            -1015854730                  use technical documentation          1      1  2019  Luxembourg           M1204         K
            -1015854730  international financial reporting standards          1      1  2019  Luxembourg           M1204         K
            -1015854730                           financial products          1      1  2019  Luxembourg           M1204         K
            -1015854730                             computer science          1      1  2019  Luxembourg           M1204         K
            -1015854730                      bookkeeping regulations          1      1  2019  Luxembourg           M1204         K
            -1015854730                             Microsoft Access          1      1  2019  Luxembourg           M1204         K
            -1015854730                          tracking principles          1      1  2019  Luxembourg           M1204         K
            -1015854730                    use spreadsheets software          1      1  2019  Luxembourg           M1204         K
            -1015854730                                      betting          1      1  2019  Luxembourg           M1204         K
            -1015854730                 use word processing software          1      5  2019  Luxembourg           C1202         K


Examples where canton differs within a vacancy:


skill  positions  month  year            canton occupation_code nace_code
source_file vacancy_id                                                                                                       
2019-2020   -1015849354      cooperate with professionals          1      5  2019          Capellen           F1106         M
            -1015849354                       mathematics          1      5  2019          Capellen           F1106         M
            -1015849354                     work in teams          1      5  2019          Capellen           F1106         M
            -1015849354                             logic          1      5  2019          Capellen           F1106         M
            -1015849354                     communication          1      5  2019          Capellen           F1106         M
            -1015849354           ICT networking hardware          1      5  2019          Capellen           F1106         M
            -1015849354         monitor construction site          1      5  2019          Capellen           F1106         M
            -1015849354                     advise others          1      5  2019          Capellen           F1106         M
            -1015849354             construction industry          1      5  2019          Capellen           F1106         M
            -1015849354       work in an organised manner          1      5  2019          Capellen           F1106         M
            -1015849354                            French          7      1  2019  Esch-sur-Alzette           N4301         H
            -1015849354         use presentation software          7      1  2019  Esch-sur-Alzette           N4301         H
            -1015849354              security regulations          7      1  2019  Esch-sur-Alzette           N4301         H
            -1015849354                work independently          7      1  2019  Esch-sur-Alzette           N4301         H
            -1015849354                    work in shifts          7      1  2019  Esch-sur-Alzette           N4301         H
            -1015840394                        accounting          1      1  2019        Luxembourg           C1202         N
            -1015840394              use microsoft office          1      1  2019        Luxembourg           C1202         N
            -1015840394               risk identification          1      1  2019        Luxembourg           C1202         N
            -1015840394             perform risk analysis          1      1  2019        Luxembourg           C1202         N
            -1015840394  show professional responsibility          1      1  2019        Luxembourg           C1202         N


Examples where occupation_code differs within a vacancy:


skill  positions  month  year      canton occupation_code nace_code
source_file vacancy_id                                                                                                            
2019-2020   -1015854730                         use microsoft office          1      1  2019  Luxembourg           M1204         K
            -1015854730                          computer technology          1      1  2019  Luxembourg           M1204         K
            -1015854730                    use presentation software          1      1  2019  Luxembourg           M1204         K
            -1015854730               geographic information systems          1      1  2019  Luxembourg           M1204         K
            -1015854730                        product comprehension          1      1  2019  Luxembourg           M1204         K
            -1015854730                                assertiveness          1      1  2019  Luxembourg           M1204         K
            -1015854730                                   accounting          1      1  2019  Luxembourg           M1204         K
            -1015854730                                    economics          1      1  2019  Luxembourg           M1204         K
            -1015854730                                  lead others          1      1  2019  Luxembourg           M1204         K
            -1015854730                                 report facts          1      1  2019  Luxembourg           M1204         K
            -1015854730                  use technical documentation          1      1  2019  Luxembourg           M1204         K
            -1015854730  international financial reporting standards          1      1  2019  Luxembourg           M1204         K
            -1015854730                           financial products          1      1  2019  Luxembourg           M1204         K
            -1015854730                             computer science          1      1  2019  Luxembourg           M1204         K
            -1015854730                      bookkeeping regulations          1      1  2019  Luxembourg           M1204         K
            -1015854730                             Microsoft Access          1      1  2019  Luxembourg           M1204         K
            -1015854730                          tracking principles          1      1  2019  Luxembourg           M1204         K
            -1015854730                    use spreadsheets software          1      1  2019  Luxembourg           M1204         K
            -1015854730                                      betting          1      1  2019  Luxembourg           M1204         K
            -1015854730                 use word processing software          1      5  2019  Luxembourg           C1202         K


Examples where nace_code differs within a vacancy:


skill  positions  month  year            canton occupation_code nace_code
source_file vacancy_id                                                                                                       
2019-2020   -1015849354      cooperate with professionals          1      5  2019          Capellen           F1106         M
            -1015849354                       mathematics          1      5  2019          Capellen           F1106         M
            -1015849354                     work in teams          1      5  2019          Capellen           F1106         M
            -1015849354                             logic          1      5  2019          Capellen           F1106         M
            -1015849354                     communication          1      5  2019          Capellen           F1106         M
            -1015849354           ICT networking hardware          1      5  2019          Capellen           F1106         M
            -1015849354         monitor construction site          1      5  2019          Capellen           F1106         M
            -1015849354                     advise others          1      5  2019          Capellen           F1106         M
            -1015849354             construction industry          1      5  2019          Capellen           F1106         M
            -1015849354       work in an organised manner          1      5  2019          Capellen           F1106         M
            -1015849354                            French          7      1  2019  Esch-sur-Alzette           N4301         H
            -1015849354         use presentation software          7      1  2019  Esch-sur-Alzette           N4301         H
            -1015849354              security regulations          7      1  2019  Esch-sur-Alzette           N4301         H
            -1015849354                work independently          7      1  2019  Esch-sur-Alzette           N4301         H
            -1015849354                    work in shifts          7      1  2019  Esch-sur-Alzette           N4301         H
            -1015840394                        accounting          1      1  2019        Luxembourg           C1202         N
            -1015840394              use microsoft office          1      1  2019        Luxembourg           C1202         N
            -1015840394               risk identification          1      1  2019        Luxembourg           C1202         N
            -1015840394             perform risk analysis          1      1  2019        Luxembourg           C1202         N
            -1015840394  show professional responsibility          1      1  2019        Luxembourg           C1202         N

## 5. Does any vacancy_id appear in more than one file?

If an ID appears in several files, it could be the same vacancy running across periods, or two different vacancies sharing an ID.

In [8]:
files_per_id = vac.groupby('vacancy_id')['source_file'].nunique()
multi_ids = files_per_id[files_per_id > 1]

print(f'Unique vacancy_ids overall: {len(files_per_id):,}')
print(f'IDs found in more than one file: {len(multi_ids):,} '
      f'({len(multi_ids) / len(files_per_id) * 100:.3f}%)')

if len(multi_ids):
    print('\nNumber of files per repeated ID:')
    display(multi_ids.value_counts().sort_index())

    print('Which file pairs share IDs:')
    shared = vac[vac['vacancy_id'].isin(multi_ids.index)]
    presence = pd.crosstab(shared['vacancy_id'], shared['source_file'].astype(str)) > 0
    pairs = presence.dot(presence.columns + ' + ').str.rstrip(' + ')
    display(pairs.value_counts().head(10))

    print('Examples (one row per vacancy per file):')
    example_ids = multi_ids.index[:5]
    display(vac[vac['vacancy_id'].isin(example_ids)]
            .sort_values(['vacancy_id', 'source_file'])
            [['vacancy_id', 'source_file', 'month', 'year', 'positions',
              'canton', 'occupation_code', 'nace_code']])

Unique vacancy_ids overall: 322,735
IDs found in more than one file: 8,224 (2.548%)

Number of files per repeated ID:


source_file
2    8162
3      62
Name: count, dtype: int64

Which file pairs share IDs:


2017-2018 + 2021-2022                3407
2019-2020 + 2021-2022                1851
2017-2018 + 2019-2020                1407
2019-2020 + 2023-2024                1119
2015-2016 + 2019-2020                 378
2017-2018 + 2019-2020 + 2021-2022      62
Name: count, dtype: int64

Examples (one row per vacancy per file):


,vacancy_id,source_file,month,year,positions,canton,occupation_code,nace_code
796227,-2139946171,2019-2020,6,2020,2,Non précisé,M1806,J
1839051,-2139946171,2023-2024,4,2023,1,Luxembourg,M1805,J
1094895,-2139946164,2019-2020,8,2019,1,Mersch,M1607,G
1661759,-2139946164,2021-2022,8,2022,1,Luxembourg,M1402,A + B + C
585943,-2139946138,2017-2018,8,2017,1,Luxembourg,M1705,K
741294,-2139946138,2019-2020,12,2020,1,Luxembourg,M1805,M
1050309,-2139946127,2019-2020,4,2019,1,Luxembourg,M1707,J
1562300,-2139946127,2021-2022,5,2022,2,Luxembourg,M1609,Q
630893,-2139946124,2017-2018,12,2018,1,Luxembourg,M1402,M
1472728,-2139946124,2021-2022,2,2022,1,Luxembourg,M1205,K


## 6. Duplicates and missing values

**Exact duplicates**: rows identical in all 11 original columns. **Repeated skill**: the same skill listed more than once for the same vacancy.

In [9]:
dup_exact = df.duplicated(subset=['source_file'] + ORIGINAL_COLS)
dup_skill = df['skill_uri'].notna() & df.duplicated(subset=['source_file', 'vacancy_id', 'skill_uri'])

dups = pd.DataFrame({
    'rows': df.groupby('source_file', observed=True).size(),
    'exact_duplicate_rows': dup_exact.groupby(df['source_file'], observed=True).sum(),
    'repeated_skill_rows': dup_skill.groupby(df['source_file'], observed=True).sum(),
})
dups['exact_dup_pct'] = (dups['exact_duplicate_rows'] / dups['rows'] * 100).round(3)
display(dups)

if dup_exact.any():
    print('Example of exact duplicate rows:')
    first_dup = df[dup_exact].iloc[0]
    display(df[(df['source_file'] == first_dup['source_file']) &
               (df['vacancy_id'] == first_dup['vacancy_id'])].head(15))

,rows,exact_duplicate_rows,repeated_skill_rows,exact_dup_pct
source_file,,,,
2015-2016,271257,8570,8570,3.159
2017-2018,424157,13052,13052,3.077
2019-2020,431048,13513,13585,3.135
2021-2022,632035,24148,24780,3.821
2023-2024,581239,21023,21023,3.617
2025-2026,545357,18350,18988,3.365


Example of exact duplicate rows:


,vacancy_id,skill,skill_uri,positions,month,year,canton,occupation_code,occupation_label,nace_code,nace_label,source_file
51,1634062409,operating systems,http://data.europa.eu/esco/skill/f9a6f35b-01a7...,1,4,2015,Luxembourg,M1805,Études et développement informatique,O,ADMINISTRATION PUBLIQUE,2015-2016
52,1634062409,project commissioning,http://data.europa.eu/esco/skill/4ca91852-c534...,1,4,2015,Luxembourg,M1805,Études et développement informatique,O,ADMINISTRATION PUBLIQUE,2015-2016
53,1634062409,concepts of telecommunications,http://data.europa.eu/esco/skill/f1a56786-e6d7...,1,4,2015,Luxembourg,M1805,Études et développement informatique,O,ADMINISTRATION PUBLIQUE,2015-2016
54,1634062409,attend to detail,http://data.europa.eu/esco/skill/83e6510b-ffeb...,1,4,2015,Luxembourg,M1805,Études et développement informatique,O,ADMINISTRATION PUBLIQUE,2015-2016
55,1634062409,computer programming,http://data.europa.eu/esco/skill/21d2f96d-35f7...,1,4,2015,Luxembourg,M1805,Études et développement informatique,O,ADMINISTRATION PUBLIQUE,2015-2016
56,1634062409,perform clerical duties,http://data.europa.eu/esco/skill/ab986bc4-df92...,1,4,2015,Luxembourg,M1805,Études et développement informatique,O,ADMINISTRATION PUBLIQUE,2015-2016
57,1634062409,database,http://data.europa.eu/esco/skill/43ae58b9-5e56...,1,4,2015,Luxembourg,M1805,Études et développement informatique,O,ADMINISTRATION PUBLIQUE,2015-2016
58,1634062409,think creatively,http://data.europa.eu/esco/skill/c624c6a3-b0ba...,1,4,2015,Luxembourg,M1805,Études et développement informatique,O,ADMINISTRATION PUBLIQUE,2015-2016
59,1634062409,project management,http://data.europa.eu/esco/skill/7111b95d-0ce3...,1,4,2015,Luxembourg,M1805,Études et développement informatique,O,ADMINISTRATION PUBLIQUE,2015-2016
60,1634062409,communication,http://data.europa.eu/esco/skill/15d76317-c71a...,1,4,2015,Luxembourg,M1805,Études et développement informatique,O,ADMINISTRATION PUBLIQUE,2015-2016


In [10]:
# Missing values per column, as a percentage of rows, for each file.
missing = (df[ORIGINAL_COLS].isna()
             .groupby(df['source_file'], observed=True).mean()
             .mul(100).round(2))
print('Missing values (% of rows):')
display(missing.T)

# skill and skill_uri should be missing together.
mismatch = (df['skill'].isna() != df['skill_uri'].isna()).sum()
print(f'Rows where skill and skill_uri disagree on missingness: {mismatch:,}')

Missing values (% of rows):


source_file,2015-2016,2017-2018,2019-2020,2021-2022,2023-2024,2025-2026
vacancy_id,0.00,0.00,0.00,0.00,0.00,0.00
skill,5.78,2.44,2.02,1.39,1.22,0.84
skill_uri,5.78,2.44,2.02,1.39,1.22,0.84
positions,0.00,0.00,0.00,0.00,0.00,0.00
month,0.00,0.00,0.00,0.00,0.00,0.00
year,0.00,0.00,0.00,0.00,0.00,0.00
canton,5.89,0.03,0.06,0.04,0.10,0.11
occupation_code,0.00,0.00,0.04,0.00,0.00,0.00
occupation_label,0.00,0.00,0.04,0.00,0.01,0.00
nace_code,1.49,1.49,1.17,0.80,0.85,1.12


Rows where skill and skill_uri disagree on missingness: 0


In [11]:
# Vacancies with no skill at all vs. vacancies with some missing skill rows.
skill_status = (df.assign(has_skill=df['skill'].notna())
                  .groupby(['source_file', 'vacancy_id'], observed=True)['has_skill']
                  .agg(['any', 'all']))
skill_summary = pd.DataFrame({
    'vacancies': skill_status.groupby(level='source_file', observed=True).size(),
    'no_skill_at_all': (~skill_status['any']).groupby(level='source_file', observed=True).sum(),
    'some_rows_missing_skill': (skill_status['any'] & ~skill_status['all'])
                                 .groupby(level='source_file', observed=True).sum(),
})
skill_summary['no_skill_pct'] = (skill_summary['no_skill_at_all'] /
                                 skill_summary['vacancies'] * 100).round(2)
display(skill_summary)

,vacancies,no_skill_at_all,some_rows_missing_skill,no_skill_pct
source_file,,,,
2015-2016,46301,15690,0,33.89
2017-2018,56405,10356,0,18.36
2019-2020,51863,8698,13,16.77
2021-2022,67012,8624,177,12.87
2023-2024,60088,7099,0,11.81
2025-2026,49352,4438,130,8.99


In [12]:
# Codes and labels should match one-to-one.
for code_col, label_col in [('occupation_code', 'occupation_label'), ('nace_code', 'nace_label')]:
    pairs = df[[code_col, label_col]].dropna().drop_duplicates()
    codes_multi = pairs.groupby(code_col, observed=True)[label_col].nunique()
    labels_multi = pairs.groupby(label_col, observed=True)[code_col].nunique()
    print(f'{code_col}: {pairs[code_col].nunique()} codes, '
          f'{(codes_multi > 1).sum()} with more than one label, '
          f'{(labels_multi > 1).sum()} labels with more than one code')

print('\nCanton values and row counts:')
display(df['canton'].value_counts(dropna=False))

occupation_code: 446 codes, 0 with more than one label, 0 labels with more than one code
nace_code: 18 codes, 0 with more than one label, 0 labels with more than one code

Canton values and row counts:


canton
Luxembourg          2030514
Esch-sur-Alzette     422545
Capellen             116863
Grevenmacher          66797
Mersch                65030
Diekirch              39023
Clervaux              35167
Remich                28312
Wiltz                 20568
Echternach            19812
NaN                   17852
Redange               12657
Non précisé            6690
Vianden                3263
Name: count, dtype: int64

## Results

### Size

- 2,885,093 rows in 6 files. One row = one vacancy–skill pair.
- 331,021 vacancies (counted per file), 322,735 unique `vacancy_id`, 449,487 posts.
- Ads list more skills over time: 5.45 rows per vacancy in 2015, 11.17 in 2026. This is why raw skill counts can't be compared between years, only shares.

| Year | Rows | Vacancies | Posts |
|---|---|---|---|
| 2015 | 118,117 | 21,672 | 31,404 |
| 2016 | 153,140 | 24,629 | 34,012 |
| 2017 | 196,113 | 27,104 | 35,738 |
| 2018 | 228,044 | 29,301 | 40,782 |
| 2019 | 227,218 | 28,175 | 39,451 |
| 2020 | 203,830 | 23,688 | 34,325 |
| 2021 | 302,572 | 32,629 | 44,994 |
| 2022 | 329,463 | 34,383 | 47,814 |
| 2023 | 280,051 | 30,003 | 40,135 |
| 2024 | 301,188 | 30,085 | 38,026 |
| 2025 | 321,000 | 29,270 | 37,260 |
| 2026 | 224,357 | 20,082 | 25,546 |

Each file only contains the two years in its name.

### Months

- 2015 to 2025: all 12 months present, no invalid month values.
- 2026: January to August are normal (about 1,600 to 2,800 vacancies per month). September has only 895, October 1, November and December 0. 2026 is complete only up to August.
- Normal patterns: August and December are always low (holidays). March to May 2020 drop to about 1,100 to 1,600 (COVID).

### Do vacancy details change within a vacancy?

- `year` never differs.
- `month` differs in 1,505 vacancies (0.46%), `occupation_code` 1,479 (0.45%), `nace_code` 1,381 (0.42%), `canton` 917 (0.28%), `positions` 308 (0.09%).
- This only happens in 3 files: 2019-2020, 2021-2022 and 2025-2026. The other files have 0.
- The examples look like two different ads that share the same ID in the same file, not one ad that changed.

### IDs in more than one file

- 8,224 IDs (2.5%) appear in 2 files, and 62 of them in 3 files.
- They are not the same vacancy running on: many file pairs are not next to each other (3,407 IDs are shared by 2017-2018 and 2021-2022), and the examples have different years, occupations and sectors.
- So IDs are reused. A vacancy must be identified by `source_file` + `vacancy_id` (plus its details), not by `vacancy_id` alone.

### Duplicates

- 98,656 exact duplicate rows (3.4%), 3.1% to 3.8% per file. In the example, a whole vacancy's skill list is repeated, which adds no information.
- Repeated skills are almost all exact duplicates. The small extra (2019-2020, 2021-2022, 2025-2026) comes from the vacancies that share an ID.

### Vacancies with no skill

- 54,905 vacancies (16.6%) have no skill at all.
- This falls over time: 33.9% in 2015-2016, 18.4% in 2017-2018, 16.8% in 2019-2020, 12.9% in 2021-2022, 11.8% in 2023-2024, 9.0% in 2025-2026. The skill detection got better.
- Vacancies with only some rows missing a skill: 320, all in the files where IDs are shared.

### Missing values (% of rows)

- `skill` / `skill_uri`: 5.78% in 2015-2016, falling to 0.84% in 2025-2026. They are always missing together.
- `canton`: 5.89% in 2015-2016, then 0.03% to 0.11%. A further 6,690 rows say "Non précisé", which also means unknown.
- `nace_code` / `nace_label`: 0.8% to 1.5% in every file.
- `occupation_code` / `occupation_label`: almost never missing (at most 0.04%).
- `vacancy_id`, `positions`, `month`, `year`: never missing.
- Codes and labels match one to one: 446 occupation codes and 18 NACE sectors.

### What this means for cleaning (notebook 02)

1. Drop exact duplicate rows.
2. Treat missing canton and "Non précisé" both as "Unknown".
3. Identify a vacancy by `source_file`, `vacancy_id`, year, month, occupation, sector, canton and positions together.
4. Use data only up to August 2026 and mark later months as incomplete.
5. Compare skills as a share of vacancies that have at least one skill, because the no-skill rate and the number of skills per ad both change over the years.
